(course-alzheimer-peptide-synthesis)=
# Path A - Module 24: Peptide Synthesis

In this module, synthesis means constructing a molecular coordinate model from
an amino-acid sequence. We use KLVFF as the five-amino-acid example for this path.
Construction does not establish therapeutic activity, binding or an experimentally
observed conformation.

The default `msm.build.build_peptide()` engine is MolSysMT. It uses bundled
construction references to supply atoms, bonds and approximate initial coordinates.
The optional AmberTools engine is selected explicitly with `engine='LEaP'`;
it is not needed for these examples. Native construction does not establish a
complete chemical state, parameterization or an energy-minimized structure.

:::{admonition} 🎯 Learning Outcomes
:class: dropdown

- Build an initial KLVFF model without a network download.
- Represent ACE/NME caps explicitly in the input sequence.
- Set and measure declared backbone torsions through existing public tools.
- Inspect the resulting coordinate model without treating it as a prediction.
:::


In [1]:
import numpy as np
import molsysmt as msm
from molsysmt import pyunitwizard as puw


(course-alzheimer-peptide-synthesis-building)=
## 1. Simple Synthesis

The one-letter sequence gives a five-group initial model. Inspect its form and
inventory before using it in another workflow. A sequence name alone does not
select the environmental protonation or terminal-charge state.


In [2]:
peptide = msm.build.build_peptide('KLVFF')
print('Peptide form:', msm.get_form(peptide))
print('Group names:', msm.get(peptide, element='group', group_name=True))
msm.info(peptide)


Peptide form: molsysmt.MolSys
Group names: ['LYS', 'LEU', 'VAL', 'PHE', 'PHE']


form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_peptides,n_structures
molsysmt.MolSys,97,5,1,1,1,1,1,1


(course-alzheimer-peptide-synthesis-capping)=
## 2. Protecting the Ends (Cappings)

To include an acetyl ACE group and an N-methylamide NME group, put their labels
in the ordered three-letter input sequence. Both caps become actual groups of
the constructed model. Choosing caps is an explicit modeling decision; it does
not by itself determine formal charges, pH behavior or degradation resistance.
The unsupported `n_terminal` and `c_terminal` builder keywords are not needed.


In [3]:
capped_peptide = msm.build.build_peptide('AceLysLeuValPhePheNme')
print('Capped atom count:', msm.get(capped_peptide, n_atoms=True))
print('Capped groups:', msm.get(capped_peptide, element='group', group_name=True))
msm.info(capped_peptide, element='group')


Capped atom count: 109
Capped groups: ['ACE', 'LYS', 'LEU', 'VAL', 'PHE', 'PHE', 'NME']


index,id,name,type,n atoms,component index,chain index,molecule index,molecule type,entity index,entity name
0,0,ACE,terminal capping,6,0,0,0,peptide,0,peptide
1,1,LYS,amino acid,22,0,0,0,peptide,0,peptide
2,2,LEU,amino acid,19,0,0,0,peptide,0,peptide
3,3,VAL,amino acid,16,0,0,0,peptide,0,peptide
4,4,PHE,amino acid,20,0,0,0,peptide,0,peptide
5,5,PHE,amino acid,20,0,0,0,peptide,0,peptide
6,6,NME,terminal capping,6,0,0,0,peptide,0,peptide


(course-alzheimer-peptide-synthesis-torsions)=
## 3. Choosing the Secondary Structure

We can assign initial backbone torsions in a helical region rather than request
an unsupported `configuration` keyword. Here the declared teaching targets are
**phi = -60 degrees**, **psi = -45 degrees** and **omega = 180 degrees**.
They are geometric inputs, not a prediction of a stable alpha-helical population.
The native side-chain placements and all other modeling choices remain initial
approximations; no force-field relaxation or secondary-structure assignment runs.

The caps provide neighboring atoms for all five phi and psi angles. Quartets
come from the stored graph in `chemical_states`; geometric edits belong to
`structure`. Copy the source first, then edit that independent copy in place.
All examples use Cartesian coordinates with `pbc=False`.


In [4]:
backbone_quartets = {
    name: msm.chemical_states.get_dihedral_quartets(capped_peptide, **{name: True})
    for name in ('phi', 'psi', 'omega')
}
target_angles = {'phi': '-60 degrees', 'psi': '-45 degrees', 'omega': '180 degrees'}
helical_peptide = msm.copy(capped_peptide)
for name, quartets in backbone_quartets.items():
    msm.structure.set_dihedral_angles(
        helical_peptide, dihedral_quartets=quartets, angles=target_angles[name],
        pbc=False, in_place=True)


### Checking the achieved angles

Measure the final geometry instead of relying on the requested label. Return
quantities follow your PyUnitWizard policy; the numerical presentation below
explicitly converts to degrees. Angles at +180 and -180 degrees are equivalent,
so the check uses circular differences. The source capped model stays unchanged.


In [5]:
measured_angles = {}
for name, quartets in backbone_quartets.items():
    measured_angles[name] = msm.structure.get_dihedral_angles(
        helical_peptide, dihedral_quartets=quartets, pbc=False)
    observed = puw.get_value(measured_angles[name], to_unit='radians')
    target = puw.get_value(puw.quantity(target_angles[name]), to_unit='radians')
    difference = observed - target
    np.testing.assert_allclose(np.arctan2(np.sin(difference), np.cos(difference)),
                               0.0, atol=1e-7)
    print(name, puw.get_value(measured_angles[name], to_unit='degrees').round(2).tolist(),
          'degrees')


phi [[-60.0, -60.0, -60.0, -60.0, -60.0]] degrees
psi [[-45.0, -45.0, -45.0, -45.0, -45.0]] degrees
omega [[-180.0, 180.0, 180.0, 180.0, 180.0]] degrees


### Inspecting the coordinate model

Explore the capped model after the declared rotations. The documentation scene
shows the same atoms and coordinates calculated above. Its shape illustrates
the chosen torsions; it does not establish favorable interactions or an energy
minimum. Further physical preparation requires a separately declared protocol.


In [6]:
molsysviewer_htmlfile = '_static/views/course_alzheimer_peptide_helix.html'


In [7]:
msm.view(helical_peptide)


'<iframe src="../../../_static/views/course_alzheimer_peptide_helix.html" width="100%" height="480px"\n        style="border:none;"></iframe>'

(course-alzheimer-peptide-synthesis-challenge)=
## 🏆 Path A Challenge: The Nano-Engineer

1. Build the longer KLVFFAL sequence as an initial coordinate model.
2. Include ACE and NME explicitly in its sequence and inspect the resulting groups.
3. Request illustrative extended-region torsions: phi = -135 degrees,
   psi = +135 degrees and omega = 180 degrees. Measure them afterwards; these
   inputs do not establish the equilibrium conformation of the free peptide or
   its conformation in an amyloid assembly.
4. Use `msm.get(..., n_atoms=True)` to obtain the capped model's atom count.

```{admonition} Key takeaway
:class: dropdown

Explicit sequence labels define the constructed inventory. Declared rotations
define an initial geometry. Chemical-state preparation, relaxation and assessment
of a candidate's behavior remain separate tasks.
```

(course-alzheimer-peptide-synthesis-see-also)=
:::{seealso}
:class: dropdown

**API documentation**

- {func}`molsysmt.build.build_peptide` — Native or explicitly selected engine construction.
- {func}`molsysmt.chemical_states.get_dihedral_quartets` — Available graph-defined quartets.
- {func}`molsysmt.structure.set_dihedral_angles` — Setting angular quantities on movable blocks.
- {func}`molsysmt.structure.get_dihedral_angles` — Measuring the resulting torsions.
- {func}`molsysmt.basic.copy` — Independent molecular-system copies.

**Related modules and guides**

- {ref}`Conformational Engineering <course-alzheimer-conformational-engineering>`.
- {ref}`Build peptide <Tutorial_Build_peptide>`.
- Next: {ref}`Module 25: Solvation Engineering <course-alzheimer-solvation-engineering>`.
:::
